# 210 — TraceAgent
Specialist agent for retrieving and explaining investigation traces stored in Neo4j.

Two trace types are in play throughout this notebook:
- **Target trace** — a historical investigation record being retrieved/summarised.
- **Operational trace** — TraceAgent's own activity log for the current session.

These are always separate objects. The recursion guard enforces this at runtime.

Supported tasks: `retrieve_trace`, `find_traces_by_entity`,
`summarize_trace`, `retrieve_and_summarize_trace`.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import uuid
import logging
import pandas as pd

from src.config import get_neo4j_settings
from src.storage.neo4j_repository import Neo4jRepository
from src.storage.trace_repository import TraceRepository
from src.tracing.trace_service import TraceService
from src.tools.graph_tools import GraphTools
from src.tools.trace_tools import TraceTools
from src.agents.graph_agent import GraphAgent
from src.agents.trace_agent import TraceAgent
from src.domain.models import InvestigationRequest, UserContext

logging.basicConfig(
    level=logging.INFO,
    format="%(name)-32s %(levelname)-8s %(message)s",
)

settings = get_neo4j_settings()
neo4j    = Neo4jRepository(**vars(settings))
neo4j.test_connection()

repo          = TraceRepository(neo4j)
trace_service = TraceService(repo)
graph_tools   = GraphTools(neo4j)
trace_tools   = TraceTools(repo)
print("Connected")

Connected


## Create test data
Run a GraphAgent investigation so there is a persisted trace to retrieve.
This simulates what a prior investigation session would have produced.

In [3]:
COMPANY = "TELEFONICA O2 HOLDINGS LIMITED"

graph_agent = GraphAgent(tools=graph_tools, trace_service=trace_service)

req_data = InvestigationRequest(
    entity_name=COMPANY,
    context=UserContext(
        user_id="analyst-001",
        session_id=str(uuid.uuid4()),
        metadata={"mode": "interactive"},
    ),
    request_id=str(uuid.uuid4()),
)
trace_data = trace_service.start_trace(req_data, req_data.context)

for task, ctx in [
    ("company_profile",      {"company_name": COMPANY}),
    ("expand_ownership",     {"company_name": COMPANY, "max_depth": 3}),
    ("shared_address_check", {"company_name": COMPANY}),
]:
    r = graph_agent.run(task, ctx, trace_data)
    print(f"[{'✓' if r.success else '✗'}] {task:<25} {r.summary[:80]}")

trace_service.finalize_trace(trace_data, final_summary=f"Graph investigation of {COMPANY} complete.")
TARGET_TRACE_ID = trace_data.request_id
print(f"\nTarget trace_id: {TARGET_TRACE_ID}")

[✓] company_profile           TELEFONICA O2 HOLDINGS LIMITED (#05310128, Active). SIC codes: 61900 (Other tele
[✓] expand_ownership          Found 1 ownership hop(s) across depths [1] for 'TELEFONICA O2 HOLDINGS LIMITED'.
[✓] shared_address_check      'TELEFONICA O2 HOLDINGS LIMITED' shares its address (BN99 3HH) with 131 other co

Target trace_id: 5f99b9dc-2e9b-48a2-b49e-4aecf44c4f6b


## Instantiate TraceAgent
The agent gets its own **operational trace** — completely separate from `trace_data`.

In [4]:
trace_agent = TraceAgent(tools=trace_tools, trace_service=trace_service)
print(f"agent : {trace_agent.name}")
print(f"ai    : {trace_agent._ai_client}")

# Operational trace — TraceAgent logs its own activity here.
req_ops = InvestigationRequest(
    entity_name="TraceAgent session",
    context=UserContext(
        user_id="analyst-001",
        session_id=str(uuid.uuid4()),
        metadata={"mode": "interactive"},
    ),
    request_id=str(uuid.uuid4()),
)
trace_ops = trace_service.start_trace(req_ops, req_ops.context)
print(f"\nOperational trace_id : {trace_ops.request_id}")
print(f"Target    trace_id   : {TARGET_TRACE_ID}")
print(f"Are they the same?   : {trace_ops.request_id == TARGET_TRACE_ID}")

agent : trace-agent
ai    : None

Operational trace_id : ee5217b4-03ad-473b-8774-29af57d9664a
Target    trace_id   : 5f99b9dc-2e9b-48a2-b49e-4aecf44c4f6b
Are they the same?   : False


## Task: retrieve_trace

In [5]:
r_retrieve = trace_agent.run(
    "retrieve_trace",
    {"trace_id": TARGET_TRACE_ID},
    trace_ops,
)
print(f"success  : {r_retrieve.success}")
print(f"summary  : {r_retrieve.summary}")

if r_retrieve.findings.get("retrieve_trace"):
    td = r_retrieve.findings["retrieve_trace"]
    print(f"\nquery      : {td['query']}")
    print(f"mode       : {td['mode']}")
    print(f"started_at : {td['started_at']}")
    print(f"ended_at   : {td['ended_at']}")
    print(f"events     : {len(td['events'])}")
    df = pd.DataFrame(td["events"])
    display(df[["event_number", "event_type", "tool_name", "output_summary", "decision"]])

success  : True
summary  : Trace '5f99b9dc-2e9b-48a2-b49e-4aecf44c4f6b' for 'TELEFONICA O2 HOLDINGS LIMITED': 3 event(s).

query      : TELEFONICA O2 HOLDINGS LIMITED
mode       : interactive
started_at : 2026-03-22T05:19:04.392331+00:00
ended_at   : 2026-03-22T05:19:04.728616+00:00
events     : 3


,event_number,event_type,tool_name,output_summary,decision
0,1,tool_returned,company_profile,"TELEFONICA O2 HOLDINGS LIMITED (#05310128, Act...",result available for downstream agents
1,2,tool_returned,expand_ownership,Found 1 ownership hop(s) across depths [1] for...,result available for downstream agents
2,3,tool_returned,shared_address_check,'TELEFONICA O2 HOLDINGS LIMITED' shares its ad...,result available for downstream agents


## Task: find_traces_by_entity

In [6]:
r_find = trace_agent.run(
    "find_traces_by_entity",
    {"entity_name": COMPANY},
    trace_ops,
)
print(f"success  : {r_find.success}")
print(f"summary  : {r_find.summary}")

if r_find.findings.get("find_traces_by_entity"):
    rows = r_find.findings["find_traces_by_entity"]
    print(f"\nFound {len(rows)} trace(s):")
    for row in rows:
        print(f"  {row['trace_id']}  started={row['started_at']}")

success  : True
summary  : Found 1 trace(s) connected to 'TELEFONICA O2 HOLDINGS LIMITED'.

Found 1 trace(s):
  5f99b9dc-2e9b-48a2-b49e-4aecf44c4f6b  started=2026-03-22T05:19:04.392331+00:00


## Task: summarize_trace — no AI client (template fallback)
When no AI client is configured, a deterministic one-sentence summary is produced
from the trace metadata. It is always available and never makes API calls.

In [7]:
trace_data_dict = r_retrieve.findings["retrieve_trace"]

r_sum_no_ai = trace_agent.run(
    "summarize_trace",
    {"trace_data": trace_data_dict},
    trace_ops,
)
print(f"success    : {r_sum_no_ai.success}")
print(f"ai_enriched: {r_sum_no_ai.findings['summarize_trace']['ai_enriched']}")
print(f"\nTemplate summary:")
print(r_sum_no_ai.summary)

success    : True
ai_enriched: False

Template summary:
Investigation of 'TELEFONICA O2 HOLDINGS LIMITED': 3 event(s) using company_profile, expand_ownership, shared_address_check. Started 2026-03-22T05:19:04.392331+00:00, ended 2026-03-22T05:19:04.728616+00:00. Graph investigation of TELEFONICA O2 HOLDINGS LIMITED complete.


## AI client setup
Skipped automatically if `ANTHROPIC_API_KEY` is not set.

In [8]:
import os

ai_client    = None
ai_settings  = None
trace_agent_ai = None
if os.getenv("ANTHROPIC_API_KEY"):
    from src.config import get_anthropic_settings
    from src.clients.anthropic_client import AnthropicClient
    ai_settings    = get_anthropic_settings()
    ai_client      = AnthropicClient(settings=ai_settings)
    trace_agent_ai = TraceAgent(tools=trace_tools, trace_service=trace_service, ai_client=ai_client)
    print(f"AI client ready — haiku: {ai_settings.model_haiku}")
    print(f"trace_agent_ai will log into trace_ops: {trace_ops.request_id}")
else:
    print("ANTHROPIC_API_KEY not set — AI cells will be skipped")

AI client ready — haiku: claude-haiku-4-5-20251001
trace_agent_ai will log into trace_ops: ee5217b4-03ad-473b-8774-29af57d9664a


## Task: summarize_trace — with AI (Haiku)
Uses the same `trace_data_dict` retrieved above. The AI summary is an
audit-friendly narrative built from the event log, not a restatement of
individual tool outputs.

In [9]:
r_sum_haiku = None

if trace_agent_ai is None:
    print("Skipped — no AI client")
else:
    # model=None → client default (Haiku). Logs into the shared trace_ops.
    r_sum_haiku = trace_agent_ai.run(
        "summarize_trace",
        {"trace_data": trace_data_dict},
        trace_ops,
    )
    print(f"success    : {r_sum_haiku.success}")
    print(f"ai_enriched: {r_sum_haiku.findings['summarize_trace']['ai_enriched']}")
    print(f"\nHaiku audit summary:")
    print(r_sum_haiku.summary)

httpx                            INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


success    : True
ai_enriched: True

Haiku audit summary:
An investigation was conducted on TELEFONICA O2 HOLDINGS LIMITED, an active telecommunications company (Companies House #05310128), to assess ownership structure and address-based risk factors. Three tools were deployed: company profile retrieval confirmed the entity's core business activities in telecommunications and professional services; ownership expansion analysis identified one level of corporate ownership with all chains terminating at corporate entities; and shared address verification revealed that the company's registered address (BN99 3HH) is shared with 131 other entities, of which 126 are currently active. The high degree of address co-location with 126 other active companies presents a moderate risk signal that warrants attention, as such arrangements can sometimes indicate shared serviced office facilities or potentially obfuscated corporate structures. No immediate escalation appears necessary at this stage, but

## Task: retrieve_and_summarize_trace
Single call that retrieves the target trace and summarises it.
Logs two events to the operational trace: `tool_returned` (retrieve) +
`agent_reasoning` (AI summary).

In [10]:
r_full = None

if trace_agent_ai is None:
    print("Skipped — no AI client")
else:
    # Logs into the same trace_ops — no new trace needed.
    r_full = trace_agent_ai.run(
        "retrieve_and_summarize_trace",
        {"trace_id": TARGET_TRACE_ID},
        trace_ops,
    )
    print(f"success  : {r_full.success}")
    print(f"summary  : {r_full.summary}")

httpx                            INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


success  : True
summary  : An investigation of TELEFONICA O2 HOLDINGS LIMITED was conducted on 2026-03-22 using three compliance tools. The company_profile tool confirmed it is an active telecommunications entity (Company #05310128), the expand_ownership tool traced ownership structure to one corporate hop with no concerning terminations, and the shared_address_check tool identified that its registered address (BN99 3HH) is shared with 131 other companies, including 126 that are currently active. While address co-location with numerous entities warrants routine monitoring, no immediate red flags or suspicious ownership structures were detected. Standard follow-up documentation of the shared address clustering and periodic re-checks on corporate changes would be appropriate for ongoing compliance tracking.


### Operational trace events
All TraceAgent tasks — no-AI and AI — logged into a single `trace_ops`.
Shows every event the agent recorded across the full session.

In [11]:
r = trace_tools.retrieve_trace(trace_ops.request_id)
print(f"[{'✓' if r.success else '✗'}] {r.summary}")
if r.data:
    df = pd.DataFrame(r.data["events"])
    display(df[["event_number", "agent_name", "event_type", "tool_name", "decision", "why"]])

[✓] Trace 'ee5217b4-03ad-473b-8774-29af57d9664a' for 'TraceAgent session': 6 event(s).


,event_number,agent_name,event_type,tool_name,decision,why
0,1,trace-agent,tool_returned,retrieve_trace,trace data available,
1,2,trace-agent,tool_returned,find_traces_by_entity,trace list available,
2,3,trace-agent,agent_reasoning,,Template summary generated for trace '5f99b9dc...,Investigation of 'TELEFONICA O2 HOLDINGS LIMIT...
3,4,trace-agent,agent_reasoning,,AI audit summary generated for trace '5f99b9dc...,An investigation was conducted on TELEFONICA O...
4,5,trace-agent,tool_returned,retrieve_trace,trace retrieved — proceeding to summarise,
5,6,trace-agent,agent_reasoning,,AI audit summary generated for trace '5f99b9dc...,An investigation of TELEFONICA O2 HOLDINGS LIM...


## Recursion guard demonstration
Attempting to retrieve or summarise the agent's own operational trace returns
an error instead of writing self-referential events.

In [12]:
# Attempt to retrieve the operational trace into itself.
r_self = trace_agent.run(
    "retrieve_trace",
    {"trace_id": trace_ops.request_id},  # same ID as the operational trace
    trace_ops,
)
print(f"success : {r_self.success}")
print(f"error   : {r_self.error}")

# Attempt retrieve_and_summarize_trace using the operational trace ID.
r_self2 = trace_agent.run(
    "retrieve_and_summarize_trace",
    {"trace_id": trace_ops.request_id},
    trace_ops,
)
print(f"\nretrieve_and_summarize self-ref → success={r_self2.success}  error={r_self2.error}")

success : False
error   : trace_id 'ee5217b4-03ad-473b-8774-29af57d9664a' is the agent's own operational trace. Pass a different trace_id to avoid self-referential retrieval.

retrieve_and_summarize self-ref → success=False  error=trace_id 'ee5217b4-03ad-473b-8774-29af57d9664a' is the agent's own operational trace. Pass a different trace_id to avoid self-referential retrieval.


## Error cases

In [13]:
# Unknown task
r_bad = trace_agent.run("not_a_task", {}, trace_ops)
print(f"unknown task       → success={r_bad.success}  error={r_bad.error}")

# retrieve_trace with missing trace_id
r_missing_id = trace_agent.run("retrieve_trace", {}, trace_ops)
print(f"missing trace_id   → success={r_missing_id.success}  error={r_missing_id.error}")

# retrieve_trace with non-existent ID
r_not_found = trace_agent.run(
    "retrieve_trace",
    {"trace_id": "00000000-0000-0000-0000-000000000000"},
    trace_ops,
)
print(f"trace not found    → success={r_not_found.success}  error={r_not_found.error}")

# summarize_trace with no trace_data
r_no_data = trace_agent.run("summarize_trace", {}, trace_ops)
print(f"missing trace_data → success={r_no_data.success}  error={r_no_data.error}")

unknown task       → success=False  error=Unknown task 'not_a_task'. Supported tasks: find_traces_by_entity, retrieve_and_summarize_trace, retrieve_trace, summarize_trace.
missing trace_id   → success=False  error=context must include a non-empty 'trace_id'.
trace not found    → success=False  error=No trace found with id '00000000-0000-0000-0000-000000000000'.
missing trace_data → success=False  error=context must include 'trace_data' as a non-empty dict (use retrieve_trace first, then pass result.data here).


## Cleanup — delete all traces created in this notebook

In [14]:
trace_service.finalize_trace(trace_ops)

for trace_id in [trace_data.request_id, trace_ops.request_id]:
    neo4j.run_query(
        """
        MATCH (t:InvestigationTrace {trace_id: $trace_id})
        OPTIONAL MATCH (t)-[:HAS_EVENT]->(e:TraceEvent)
        DETACH DELETE t, e
        """,
        {"trace_id": trace_id},
    )
    print(f"Deleted {trace_id}")

Deleted 5f99b9dc-2e9b-48a2-b49e-4aecf44c4f6b
Deleted ee5217b4-03ad-473b-8774-29af57d9664a


In [15]:
neo4j.close()
print("Driver closed")

Driver closed
